# The Silence of Policy
## NLP Pipeline for Context-Adjusted SDG 5 Gap Analysis

**CSSM 550 — Special Topics in Computational Social Science and Media**  
Şevval Çakıcı · Koç University · Spring 2025  
Instructor: Merih Angın

---

### Overview
This notebook implements a three-stage NLP pipeline to measure *context-adjusted policy silence* on SDG 5 (Gender Equality) across 42 countries' UN Voluntary National Reviews (VNRs).

**Pipeline:**
1. **Stage 1** — PDF extraction, gender relevance filtering, sub-target tagging
2. **Stage 2** — LLM-based commitment classification (GPT-4o-mini)
3. **Stage 3** — Gap matrix construction and statistical analysis

**Key concept:** A country's silence on a given sub-target may reflect genuine policy failure — or simply the absence of that problem domestically. This pipeline distinguishes between *meaningful silence* and *legitimate silence* by conditioning analysis on internationally verified need thresholds.

---

### Prerequisites
- VNR PDFs stored in Google Drive (42 countries, 2018–2023)
- Benchmark CSVs stored in Google Drive (9 datasets: World Bank WBL, WHO, UNICEF, ILO, IPU, UN Population Division, OECD SIGI, ITU/WDI, V-Dem)
- OpenAI API key stored as a Colab secret named `CSSM550`

## Setup
### Install Dependencies

In [ ]:
!pip install pdfplumber sentence-transformers openai tqdm -q

### Mount Google Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import re
from collections import Counter

# Update these paths to match your Drive structure
PDF_DIR    = '/content/drive/MyDrive/KU CSSM/1st Spring/CSSM550/Capstone Project/Data/VNR PDFs'
BENCH_DIR  = '/content/drive/MyDrive/KU CSSM/1st Spring/CSSM550/Capstone Project/Data/Sub-Target'
OUTPUT_DIR = '/content/drive/MyDrive/KU CSSM/1st Spring/CSSM550/Capstone Project/Data/VNR PDFs/CSV'

os.makedirs(OUTPUT_DIR, exist_ok=True)

files = os.listdir(PDF_DIR)
print(f'VNR PDFs found: {len([f for f in files if f.endswith(".pdf")])}')

## Country Sample
44 countries initially selected via stratified purposive sampling across 4 GGI quartiles × 6 geographic regions.  
Final corpus: **42 countries** after removing France (French-only VNR) and Tanzania (poor PDF quality).

In [ ]:
countries = [
    # (country, region, GGI_quartile, GGI_score_2023)
    # Regions: ENA=Europe & N.America, EAP=East Asia & Pacific,
    # LAC=Latin America & Caribbean, SSA=Sub-Saharan Africa,
    # SSA2=South & SE Asia, MENA=Middle East & N.Africa
    ('Australia',            'EAP',  'Q1', 0.788),
    ('Bahrain',              'MENA', 'Q4', 0.602),
    ('Bangladesh',           'SSA2', 'Q4', 0.718),
    ('Barbados',             'LAC',  'Q1', 0.782),
    ('Canada',               'ENA',  'Q2', 0.770),
    ('China',                'EAP',  'Q3', 0.678),
    ('Egypt',                'MENA', 'Q4', 0.625),
    ('Ethiopia',             'SSA',  'Q4', 0.630),
    ('Finland',              'ENA',  'Q1', 0.863),
    ('Germany',              'ENA',  'Q1', 0.815),
    ('Ghana',                'SSA',  'Q3', 0.676),
    ('Guyana',               'LAC',  'Q2', 0.737),
    ('Iceland',              'ENA',  'Q1', 0.912),
    ('India',                'SSA2', 'Q4', 0.643),
    ('Indonesia',            'SSA2', 'Q3', 0.697),
    ('Italy',                'ENA',  'Q3', 0.705),
    ('Japan',                'EAP',  'Q4', 0.647),
    ('Kenya',                'SSA',  'Q3', 0.684),
    ('Kuwait',               'MENA', 'Q3', 0.639),
    ('Mongolia',             'EAP',  'Q2', 0.737),
    ('Mozambique',           'SSA',  'Q2', 0.726),
    ('Namibia',              'SSA',  'Q1', 0.802),
    ('Nepal',                'SSA2', 'Q4', 0.680),
    ('Netherlands',          'ENA',  'Q3', 0.720),
    ('New_Zealand',          'EAP',  'Q1', 0.856),
    ('Nigeria',              'SSA',  'Q4', 0.625),
    ('Norway',               'ENA',  'Q1', 0.879),
    ('Pakistan',             'SSA2', 'Q4', 0.575),
    ('Papua_New_Guinea',     'EAP',  'Q4', 0.598),
    ('Philippines',          'SSA2', 'Q2', 0.791),
    ('Rwanda',               'SSA',  'Q1', 0.794),
    ('Saint_Lucia',          'LAC',  'Q3', 0.671),
    ('Saudi_Arabia',         'MENA', 'Q4', 0.602),
    ('South_Africa',         'SSA',  'Q2', 0.740),
    ('Sri_Lanka',            'SSA2', 'Q2', 0.724),
    ('Sweden',               'ENA',  'Q1', 0.815),
    ('Thailand',             'EAP',  'Q3', 0.709),
    ('Trinidad_and_Tobago',  'LAC',  'Q4', 0.680),
    ('Turkey',               'ENA',  'Q4', 0.639),
    ('UAE',                  'MENA', 'Q3', 0.630),
    ('Uganda',               'SSA',  'Q3', 0.703),
    ('Vietnam',              'SSA2', 'Q3', 0.695),
]

df_meta = pd.DataFrame(countries, columns=['country', 'region', 'quartile', 'ggi_score'])
print(f'Sample: {len(df_meta)} countries')
print(df_meta.groupby(['region', 'quartile']).size().unstack(fill_value=0))

---
## Stage 1 — Text Extraction and Tagging
### 1.1 PDF Extraction
Extract paragraphs from each VNR PDF using `pdfplumber`. Paragraphs shorter than 100 characters are discarded (page numbers, headers, etc.).

In [ ]:
import pdfplumber

def extract_paragraphs(pdf_path):
    """Extract text paragraphs from a VNR PDF."""
    paragraphs = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            chunks = re.split(r'\n{2,}', text.strip())
            for chunk in chunks:
                chunk = chunk.replace('\n', ' ').strip()
                if len(chunk) > 100:
                    paragraphs.append(chunk)
    return paragraphs

# Quick test on one file
test_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
if test_files:
    test_paras = extract_paragraphs(os.path.join(PDF_DIR, test_files[0]))
    print(f'Test ({test_files[0]}): {len(test_paras)} paragraphs extracted')
    print(f'Sample: {test_paras[0][:200]}...')

### 1.2 Gender Relevance Filter
Retain only paragraphs containing gender-related keywords. **Recall is prioritized over precision** — false negatives cannot be recovered downstream.

In [ ]:
GENDER_KEYWORDS = [
    'woman', 'women', 'girl', 'girls', 'gender', 'female',
    'maternal', 'maternity', 'feminist', 'patriarchy',
    'sdg 5', 'sdg5', 'goal 5', 'gender equality',
    'gender-based', 'gender based', 'unpaid care',
    'child marriage', 'female genital', 'fgm',
    'reproductive', 'contracepti', 'family planning',
    'violence against women', 'domestic violence',
    "women's rights", "women's empowerment",
    'gender gap', 'gender parity', 'gender mainstreaming'
]

def is_gender_relevant(text):
    """Return True if paragraph contains at least one gender keyword."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in GENDER_KEYWORDS)

### 1.3 Process All VNRs
Extract and filter all 42 VNR PDFs.

In [ ]:
all_data = []

for filename in sorted(os.listdir(PDF_DIR)):
    if not filename.endswith('.pdf'):
        continue
    # Parse country name from filename format: VNR_Report_Country_Year.pdf
    country = filename.replace('VNR_Report_', '').replace('.pdf', '')
    country = '_'.join(country.split('_')[:-1])
    pdf_path = os.path.join(PDF_DIR, filename)
    try:
        paras_all    = extract_paragraphs(pdf_path)
        paras_gender = [p for p in paras_all if is_gender_relevant(p)]
        for p in paras_gender:
            all_data.append({'country': country, 'filename': filename, 'paragraph': p})
        print(f'{country}: {len(paras_all)} → {len(paras_gender)} gender-relevant paragraphs')
    except Exception as e:
        print(f'ERROR {country}: {e}')

df = pd.DataFrame(all_data)

# Remove France (French-only) and Tanzania (poor OCR quality)
df = df[~df['country'].isin(['France', 'Tanzania'])]
print(f'\nFinal corpus: {len(df)} paragraphs, {df["country"].nunique()} countries')

### 1.4 SDG 5 Sub-Target Tagging
Tag each paragraph to one or more SDG 5 sub-targets using a keyword dictionary aligned with official UN indicator language.

In [ ]:
SUBTARGET_KEYWORDS = {
    '5.1': ['legal discrimination', 'discriminat', 'equal rights', 'legal framework',
             'legislation', 'law reform', 'legal protection', 'constitutional',
             'legal equality', 'non-discrimination', 'labor law', 'employment law'],
    '5.2': ['violence against women', 'gender-based violence', 'gbv', 'domestic violence',
             'intimate partner', 'sexual violence', 'harassment', 'abuse',
             'femicide', 'trafficking', 'exploitation', 'rape'],
    '5.3': ['child marriage', 'early marriage', 'forced marriage', 'female genital',
             'fgm', 'fgc', 'harmful practice', 'child bride', 'underage marriage'],
    '5.4': ['unpaid care', 'care work', 'domestic work', 'caregiving',
             'childcare', 'child care', 'eldercare', 'parental leave',
             'maternity leave', 'paternity leave', 'work-life balance', 'housework'],
    '5.5': ['women in parliament', 'political participation', 'leadership',
             'decision-making', 'representation', 'elected', 'women in politics',
             'gender quota', 'women in government', 'managerial position'],
    '5.6': ['reproductive health', 'reproductive rights', 'family planning',
             'contracepti', 'maternal health', 'antenatal', 'prenatal',
             'sexual health', 'unmet need', 'birth control', 'abortion'],
    '5.a': ['land rights', 'land ownership', 'property rights', 'asset',
             'inheritance', 'financial services', 'credit', 'bank account',
             'economic resources', 'land access', 'tenure'],
    '5.b': ['technology', 'ict', 'digital', 'internet access', 'mobile phone',
             'mobile internet', 'digital divide', 'e-government',
             'digital literacy', 'online', 'connectivity'],
    '5.c': ['gender policy', 'gender mainstreaming', 'gender responsive',
             'national action plan', 'gender budget', 'gender strategy',
             'gender equality policy', 'institutional mechanism', 'gender ministry',
             'gender office', 'gender data', 'gender statistics'],
}

def tag_subtargets(text):
    """Return list of matching SDG 5 sub-targets for a paragraph."""
    text_lower = text.lower()
    tags = [st for st, kws in SUBTARGET_KEYWORDS.items() if any(kw in text_lower for kw in kws)]
    return tags if tags else ['untagged']

df['subtargets']      = df['paragraph'].apply(tag_subtargets)
df['subtarget_count'] = df['subtargets'].apply(len)

# Distribution
all_tags   = [t for tags in df['subtargets'] for t in tags]
tag_counts = Counter(all_tags)
print('Tag distribution:')
for tag in ['5.1','5.2','5.3','5.4','5.5','5.6','5.a','5.b','5.c','untagged']:
    print(f'  {tag}: {tag_counts.get(tag, 0)}')

# Save
df.to_csv(os.path.join(OUTPUT_DIR, 'vnr_paragraphs_tagged.csv'), index=False)
print(f'\nSaved: {len(df)} rows')

---
## Stage 2 — Commitment Classification
### 2.1 OpenAI API Setup
The API key is stored as a Colab secret named `CSSM550`.

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('CSSM550'))

# Verify connection
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': "Say 'API connected' and nothing else."}]
)
print(test.choices[0].message.content)

### 2.2 Classification Rubric
An **actionable commitment** requires ALL THREE of:
1. A named responsible actor
2. A specified policy action
3. A temporal or measurable target

The classification includes surrounding context (previous and next paragraph from the same country) to avoid over-classifying as non-actionable when the actor is established in a nearby sentence.

In [ ]:
SYSTEM_PROMPT = """You are analyzing paragraphs from UN Voluntary National Reviews (VNRs) for gender equality commitments.

Classify each paragraph as either:
- "actionable": contains ALL THREE of (1) a named responsible actor, (2) a specified policy action, and (3) a temporal or measurable target
- "mention": references gender equality but lacks one or more of the three elements above

Respond with ONLY one word: either "actionable" or "mention". Nothing else."""

def classify_paragraph(text, context_before='', context_after=''):
    """Classify a paragraph as 'actionable' or 'mention'."""
    full_text = ''
    if context_before:
        full_text += f'[Previous paragraph]: {context_before[:200]}\n\n'
    full_text += f'[Paragraph to classify]: {text}'
    if context_after:
        full_text += f'\n\n[Next paragraph]: {context_after[:200]}'

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': full_text}
        ],
        max_tokens=5,
        temperature=0
    )
    result = response.choices[0].message.content.strip().lower()
    return result if result in ['actionable', 'mention'] else 'mention'

# Rubric validation tests
tests = [
    ("The government will establish 50 new women's shelters by 2025, funded by the Ministry of Social Affairs.", 'actionable'),
    ("Gender equality remains an important priority for sustainable development.", 'mention'),
]
for text, expected in tests:
    result = classify_paragraph(text)
    status = '✓' if result == expected else '✗'
    print(f'{status} Expected: {expected} | Got: {result}')

### 2.3 Classify Full Corpus
⚠️ **Note:** This cell takes ~25 minutes to run (2,131 paragraphs × GPT-4o-mini). Estimated cost: ~$0.08.

In [ ]:
import time
from tqdm import tqdm

classifications = []
rows = df.reset_index(drop=True)

for i, row in tqdm(rows.iterrows(), total=len(rows)):
    country = row['country']
    prev_para = rows.iloc[i-1]['paragraph'] if i > 0 and rows.iloc[i-1]['country'] == country else ''
    next_para = rows.iloc[i+1]['paragraph'] if i < len(rows)-1 and rows.iloc[i+1]['country'] == country else ''
    label = classify_paragraph(row['paragraph'], prev_para, next_para)
    classifications.append(label)
    time.sleep(0.1)  # Respect rate limits

df['classification'] = classifications

n_action = (df['classification'] == 'actionable').sum()
n_mention = (df['classification'] == 'mention').sum()
print(f'Actionable: {n_action} ({n_action/len(df)*100:.1f}%)')
print(f'Mention only: {n_mention} ({n_mention/len(df)*100:.1f}%)')
print(f'\nActionable rate by country:')
print(df.groupby('country').apply(
    lambda x: round((x['classification'] == 'actionable').mean() * 100, 1)
).sort_values(ascending=False).to_string())

df.to_csv(os.path.join(OUTPUT_DIR, 'vnr_paragraphs_classified.csv'), index=False)
print('\nSaved: vnr_paragraphs_classified.csv')

---
## Stage 3 — Context-Adjusted Gap Analysis
### 3.1 Load Benchmark Data
Load the 9 benchmark CSVs (one per SDG 5 sub-target). Each contains `country` and `high_need` columns (`yes`/`no`/`missing`).

Benchmark sources:
| Sub-target | Source | Threshold |
|---|---|---|
| 5.1 Legal discrimination | World Bank WBL 2024 | Score < 75 |
| 5.2 GBV | WHO via OWID | > 10% IPV (12-month) |
| 5.3 Child marriage | UNICEF 2023 | > 20% married by 18 |
| 5.4 Unpaid care | World Bank WDI / ILOSTAT | > 3 hr/day gender gap |
| 5.5 Parliament | IPU Parline 2024 | < 30% women |
| 5.6 Reproductive rights | World Bank WDI / UN Pop Div | > 10% unmet need |
| 5.a Land rights | OECD SIGI 2023 + FAO GLRD | SIGI score > 0.3 |
| 5.b Internet access | ITU via World Bank WDI | > 10 pp gender gap |
| 5.c Gender-responsive policy | V-Dem v16 | Score < 0.5 |

In [ ]:
benchmark_files = {
    '5.1': 'benchmark_5_1_wbl.csv',
    '5.2': 'benchmark_5_2_ipv.csv',
    '5.3': 'benchmark_5_3_childmarriage.csv',
    '5.4': 'benchmark_5_4_unpaidwork.csv',
    '5.5': 'benchmark_5_5_parliament.csv',
    '5.6': 'benchmark_5_6_unmet.csv',
    '5.a': 'benchmark_5_a_sigi.csv',
    '5.b': 'benchmark_5_b_internet.csv',
    '5.c': 'benchmark_5_c_vdem.csv',
}

benchmarks = {}
for code, fname in benchmark_files.items():
    col = f'need_{code.replace(".", "_")}'
    benchmarks[code] = pd.read_csv(os.path.join(BENCH_DIR, fname))[['country', 'high_need']].rename(columns={'high_need': col})

df_need = benchmarks['5.1']
for code in ['5.2','5.3','5.4','5.5','5.6','5.a','5.b','5.c']:
    df_need = df_need.merge(benchmarks[code], on='country', how='outer')

print(f'Need matrix: {df_need.shape}')
print(df_need.head())

### 3.2 Build Commitment Matrix

In [ ]:
import ast

# Load classified paragraphs (if starting fresh from saved CSV)
CSV_DIR = OUTPUT_DIR
df_classified = pd.read_csv(os.path.join(CSV_DIR, 'vnr_paragraphs_classified.csv'))
df_classified['subtargets_list'] = df_classified['subtargets'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

subtargets = ['5.1','5.2','5.3','5.4','5.5','5.6','5.a','5.b','5.c']
commitment_data = []

for country in df_classified['country'].unique():
    row = {'country': country}
    cdf = df_classified[df_classified['country'] == country]
    for st in subtargets:
        tagged         = cdf[cdf['subtargets_list'].apply(lambda x: st in x)]
        has_actionable = (tagged['classification'] == 'actionable').any()
        row[f'commit_{st.replace(".", "_")}'] = 'yes' if has_actionable else 'no'
    commitment_data.append(row)

df_commit = pd.DataFrame(commitment_data)
print(f'Commitment matrix: {df_commit.shape}')
print(df_commit.head())

### 3.3 Compute Gap Matrix
For each country × sub-target cell, classify into one of four categories:
- **gap** — high-need + no actionable commitment (primary outcome of interest)
- **addressed** — high-need + actionable commitment
- **proactive** — low-need + actionable commitment (norm diffusion signal)
- **not_needed** — low-need + no commitment (expected)
- **missing** — benchmark data unavailable

In [ ]:
df_matrix = df_need.merge(df_commit, on='country', how='left')

st_codes = ['5_1','5_2','5_3','5_4','5_5','5_6','5_a','5_b','5_c']

for st in st_codes:
    need   = f'need_{st}'
    commit = f'commit_{st}'
    gap    = f'gap_{st}'
    def label(row):
        n, c = row[need], row[commit]
        if n == 'yes'  and c == 'no':  return 'gap'
        if n == 'yes'  and c == 'yes': return 'addressed'
        if n == 'no'   and c == 'yes': return 'proactive'
        if n == 'no':                  return 'not_needed'
        return 'missing'
    df_matrix[gap] = df_matrix.apply(label, axis=1)

def context_gap_score(row):
    """Proportion of high-need sub-targets with no actionable commitment."""
    high_need = [row[f'gap_{st}'] for st in st_codes if row[f'gap_{st}'] in ('gap','addressed')]
    gaps      = [v for v in high_need if v == 'gap']
    return round(len(gaps)/len(high_need), 3) if high_need else None

df_matrix['gap_score'] = df_matrix.apply(context_gap_score, axis=1)

# Merge metadata
df_final = df_matrix.merge(df_meta, on='country', how='left')
df_final.to_csv(os.path.join(OUTPUT_DIR, 'vnr_gap_matrix_final.csv'), index=False)

print('Gap scores (sorted):')
print(df_final[['country','gap_score']].sort_values('gap_score', ascending=False).to_string())

### 3.4 Data Quality Check
Identify countries with too many missing benchmarks and exclude them from primary analysis.

In [ ]:
gap_cols = [f'gap_{st}' for st in st_codes]
df_final['missing_count'] = df_final[gap_cols].apply(lambda r: (r=='missing').sum(), axis=1)
df_final['valid_count']   = 9 - df_final['missing_count']

print('Valid sub-targets per country:')
print(df_final[['country','valid_count','missing_count']].sort_values('valid_count', ascending=False).to_string())
print(f'\n5+ valid sub-targets: {(df_final["valid_count"] >= 5).sum()} countries')

# Exclude countries with < 5 valid sub-targets
exclude       = ['Kuwait', 'Bahrain', 'Saint_Lucia', 'UAE']
df_analysis   = df_final[~df_final['country'].isin(exclude)].copy()
print(f'Analysis sample: {len(df_analysis)} countries')

### 3.5 Statistical Analysis
Chi-square tests per sub-target and Kruskal-Wallis tests for regional and GGI quartile variation.

In [ ]:
from scipy.stats import chi2_contingency, kruskal

# --- Per-sub-target chi-square ---
print(f'{"Sub-target":<10} {"n (high-need)":<16} {"Gap %":<10} {"Addressed %":<14} {"p-value"}')
print('-' * 60)

st_results = []
for st in st_codes:
    col = f'gap_{st}'
    hn  = df_analysis[df_analysis[col].isin(['gap','addressed'])]
    n   = len(hn)
    if n == 0: continue
    ng  = (hn[col] == 'gap').sum()
    na  = (hn[col] == 'addressed').sum()
    gp  = round(ng/n*100, 1)
    ap  = round(na/n*100, 1)
    if ng > 0 and na > 0:
        _, p, _, _ = chi2_contingency([[ng, na],[n-ng, n-na]])
        pval = f'{p:.3f}'
    else:
        pval = 'N/A'
    print(f'{st:<10} {n:<16} {gp:<10} {ap:<14} {pval}')
    st_results.append({'subtarget': st, 'n': n, 'gap_pct': gp})

print('\nRanked by gap rate:')
for r in sorted(st_results, key=lambda x: x['gap_pct'], reverse=True):
    print(f"  {r['subtarget']}: {r['gap_pct']}% (n={r['n']})")

# --- Regional and GGI quartile variation ---
df_analysis['gap_score'] = df_analysis.apply(context_gap_score, axis=1)

print('\nMean gap score by region:')
print(df_analysis.groupby('region')['gap_score'].agg(['mean','count']).round(3))

print('\nMean gap score by GGI quartile:')
print(df_analysis.groupby('quartile')['gap_score'].agg(['mean','count']).round(3))

groups_r = [g['gap_score'].dropna().values for _,g in df_analysis.groupby('region') if g['gap_score'].dropna().shape[0]>0]
groups_q = [g['gap_score'].dropna().values for _,g in df_analysis.groupby('quartile') if g['gap_score'].dropna().shape[0]>0]
stat_r, p_r = kruskal(*groups_r)
stat_q, p_q = kruskal(*groups_q)
print(f'\nKruskal-Wallis (region):   H={stat_r:.3f}, p={p_r:.3f}')
print(f'Kruskal-Wallis (quartile): H={stat_q:.3f}, p={p_q:.3f}')

---
## Visualisation
### 4.1 Context-Adjusted Gap Heatmap

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# 5-category classification for heatmap
def gap5cat(row, st):
    n, c = row[f'need_{st}'], row[f'commit_{st}']
    if n=='yes' and c=='no':  return 'gap'
    if n=='yes' and c=='yes': return 'addressed'
    if n=='no'  and c=='yes': return 'proactive'
    if n=='no':               return 'not_needed'
    return 'missing'

for st in st_codes:
    df_analysis[f'cat_{st}'] = df_analysis.apply(lambda r: gap5cat(r, st), axis=1)

color_map = {'missing':0, 'gap':1, 'addressed':2, 'not_needed':3, 'proactive':4}
sorted_df = df_analysis.sort_values(['region','gap_score'], ascending=[True, False])
matrix    = np.array([[color_map.get(row[f'cat_{st}'], 0) for st in st_codes] for _, row in sorted_df.iterrows()])

fig, ax = plt.subplots(figsize=(14, 12))
cmap = mcolors.ListedColormap(['#f0f0f0','#d73027','#1a9850','#bababa','#2166ac'])
norm = mcolors.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], cmap.N)
ax.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')

ax.set_xticks(range(9))
ax.set_xticklabels(['5.1\nLegal','5.2\nGBV','5.3\nChild\nMarr.','5.4\nUnpaid','5.5\nParl.','5.6\nReprod.','5.a\nLand','5.b\nInternet','5.c\nPolicy'], fontsize=10)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df['country'].str.replace('_',' '), fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#d73027', label='Gap (high-need, no commitment)'),
    Patch(facecolor='#1a9850', label='Addressed (high-need + commitment)'),
    Patch(facecolor='#2166ac', label='Proactive (low-need + commitment)'),
    Patch(facecolor='#bababa', label='Not needed'),
    Patch(facecolor='#f0f0f0', label='Missing data'),
], loc='upper right', bbox_to_anchor=(1.45, 1.0), fontsize=9)

ax.set_title('Context-Adjusted SDG 5 Policy Gap Matrix\n(38 countries × 9 sub-targets)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/KU CSSM/1st Spring/CSSM550/Capstone Project/heatmap_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: heatmap_v2.png')

### 4.2 Interactive Choropleth Map

In [ ]:
!pip install plotly -q
import plotly.graph_objects as go

ISO_MAP = {
    'Australia':'AUS','Bangladesh':'BGD','Barbados':'BRB','Canada':'CAN',
    'China':'CHN','Egypt':'EGY','Ethiopia':'ETH','Finland':'FIN',
    'Germany':'DEU','Ghana':'GHA','Guyana':'GUY','Iceland':'ISL',
    'India':'IND','Indonesia':'IDN','Italy':'ITA','Japan':'JPN',
    'Kenya':'KEN','Mongolia':'MNG','Mozambique':'MOZ','Namibia':'NAM',
    'Nepal':'NPL','Netherlands':'NLD','New_Zealand':'NZL','Nigeria':'NGA',
    'Norway':'NOR','Pakistan':'PAK','Papua_New_Guinea':'PNG','Philippines':'PHL',
    'Rwanda':'RWA','Saudi_Arabia':'SAU','South_Africa':'ZAF','Sri_Lanka':'LKA',
    'Sweden':'SWE','Thailand':'THA','Trinidad_and_Tobago':'TTO','Turkey':'TUR',
    'Uganda':'UGA','Vietnam':'VNM'
}

df_map = df_analysis[['country','gap_score','region','quartile']].dropna(subset=['gap_score']).copy()
df_map['iso']   = df_map['country'].map(ISO_MAP)
df_map['label'] = df_map['country'].str.replace('_',' ')

fig = go.Figure(go.Choropleth(
    locations=df_map['iso'],
    z=df_map['gap_score'],
    text=df_map['label'],
    customdata=df_map[['region','quartile','gap_score']].values,
    hovertemplate='<b>%{text}</b><br>Gap Score: <b>%{customdata[2]:.3f}</b><br>Region: %{customdata[0]}<br>GGI Quartile: %{customdata[1]}<extra></extra>',
    colorscale=[[0,'#2166ac'],[.25,'#92c5de'],[.5,'#f7f7f7'],[.75,'#f4a582'],[1,'#b2182b']],
    zmin=0, zmax=1,
    colorbar=dict(title='Gap Score', tickvals=[0,.25,.5,.75,1], ticktext=['0.0','0.25','0.50','0.75','1.0']),
    marker_line_color='white', marker_line_width=0.5,
))
fig.update_layout(
    title=dict(text='Context-Adjusted SDG 5 Policy Gap Score by Country', x=0.5, font=dict(size=15)),
    geo=dict(showframe=False, showcoastlines=True, coastlinecolor='lightgrey',
             showland=True, landcolor='#f5f5f5', showocean=True, oceancolor='#e8f4f8',
             projection_type='natural earth', resolution=50),
    height=550, margin=dict(l=0,r=0,t=60,b=0), paper_bgcolor='white',
)
fig.write_html('/content/drive/MyDrive/KU CSSM/1st Spring/CSSM550/Capstone Project/gap_map_v2.html')
fig.show()
print('Saved: gap_map_v2.html')

---
## Qualitative Validation
Close-reading of 4 outlier cases to validate automated classification and provide interpretive depth.

In [ ]:
outliers = {
    'Bangladesh':   'Positive outlier — gap score 0.0 (transformative commitment)',
    'Guyana':       'Meaningful silence — gap score 1.0',
    'Saudi_Arabia': 'Selective mainstreaming — gap score 0.8',
    'Norway':       'Norm diffusion — no domestic high-need, all mention-only',
}

for country, interpretation in outliers.items():
    print(f'\n{"="*60}')
    print(f'{country} — {interpretation}')
    print(f'{"="*60}')
    cdf = df_classified[df_classified['country'] == country]
    act = cdf[cdf['classification'] == 'actionable']
    men = cdf[cdf['classification'] == 'mention']
    print(f'Total paragraphs: {len(cdf)} | Actionable: {len(act)} | Mention-only: {len(men)}')
    if len(act) > 0:
        print(f'\nBest actionable commitment:')
        print(act.iloc[0]['paragraph'][:400])
    else:
        print(f'\nNo actionable commitments found.')
    print(f'\nSample mention-only paragraph:')
    print(men.iloc[0]['paragraph'][:300] if len(men)>0 else 'N/A')

---
## Export for Web Visualisation

In [ ]:
# Export gap scores for the interactive web tool (index.html)
web_export = df_final[['country','region','quartile','ggi_score','gap_score']]
web_export.to_csv(os.path.join(OUTPUT_DIR, 'gap_scores_web.csv'), index=False)
print('Exported gap_scores_web.csv')
print(web_export.to_string())